# SmartPrice Training Notebook
This notebook trains and evaluates multiple regression models.


In [ ]:
import pandas as pd
from pathlib import Path
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.tree import DecisionTreeRegressor
from sklearn.linear_model import LinearRegression


: 

In [ ]:
df = pd.read_csv(Path('..') / 'data' / 'dataset.csv')
df.head()


In [ ]:
X = df.drop(columns=['price'])
y = df['price']

preprocessor = ColumnTransformer(
    transformers=[
        ('cat', Pipeline([
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('onehot', OneHotEncoder(handle_unknown='ignore'))
        ]), ['brand']),
        ('num', Pipeline([
            ('imputer', SimpleImputer(strategy='median'))
        ]), ['ram','storage','processor_speed','battery_capacity','camera_mp'])
    ]
)

models = {
    'Linear Regression': LinearRegression(),
    'Random Forest': RandomForestRegressor(n_estimators=200, random_state=42),
    'Decision Tree': DecisionTreeRegressor(random_state=42)
}


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

results = []
for name, model in models.items():
    pipeline = Pipeline([('preprocessor', preprocessor), ('model', model)])
    pipeline.fit(X_train, y_train)
    preds = pipeline.predict(X_test)
    results.append({
        'model': name,
        'r2': r2_score(y_test, preds),
        'mae': mean_absolute_error(y_test, preds)
    })

pd.DataFrame(results)
